# Intérpretes y compiladores · Análisis léxico y sintáctico

Primera mitad de un lenguaje de juguete, **Guanche**: enteros, textos, variables (`sea x = 1;`), condicionales `si`/`sino`, bucles `mientras` y funciones (`funcion f(a, b) { devuelve a + b; }`). Aquí se escribe el *front end*: el analizador léxico (tokens con línea y columna), la gramática en EBNF, el árbol de sintaxis abstracta (AST) con `dataclasses`, un analizador descendente recursivo con precedencia y asociatividad, un analizador de Pratt como alternativa y un impresor que devuelve el árbol a código. Las comprobaciones se hacen contra un segundo método: un lexer con expresiones regulares y el analizador del propio Python (`ast`), que tiene la misma precedencia aritmética. La evaluación y la máquina virtual están en el notebook siguiente.

## 1. Tokens con posición

Escribir `tokenizar(codigo)`, un analizador léxico a mano que recorra el texto carácter a carácter y devuelva una lista de `Token(tipo, valor, linea, columna)`: números enteros, nombres, palabras reservadas, símbolos de uno o dos caracteres (`==`, `<=`…) y un token `FIN`. Comprobar que cada token aparece en el código justo en la línea y columna que indica.

In [1]:
import string
from dataclasses import dataclass

RESERVADAS = {"sea", "si", "sino", "mientras", "funcion", "devuelve", "verdadero", "falso", "nulo", "y", "o", "no"}
SIMBOLOS = ["==", "!=", "<=", ">=", "+", "-", "*", "/", "%", "^", "<", ">", "=", "(", ")", "{", "}", ",", ";"]
ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


@dataclass(frozen=True, slots=True)
class Token:
    tipo: str
    valor: object
    linea: int
    columna: int


class ErrorLexico(Exception):
    def __init__(self, mensaje, linea, columna):
        super().__init__(f"{linea}:{columna}: {mensaje}")
        self.linea, self.columna = linea, columna


def tokenizar(codigo):
    tokens = []
    i, linea, inicio_linea = 0, 1, 0
    while i < len(codigo):
        c = codigo[i]
        columna = i - inicio_linea + 1
        if c == "\n":
            linea, inicio_linea, i = linea + 1, i + 1, i + 1
        elif c in " \t\r":
            i += 1
        elif c == "#":
            while i < len(codigo) and codigo[i] != "\n":
                i += 1
        elif c in string.digits:
            j = i
            while j < len(codigo) and codigo[j] in string.digits:
                j += 1
            if j < len(codigo) and (codigo[j].isalpha() or codigo[j] == "_"):
                raise ErrorLexico(f"número mal formado {codigo[i:j + 1]!r}", linea, columna)
            tokens.append(Token("NUMERO", int(codigo[i:j]), linea, columna))
            i = j
        elif c.isalpha() or c == "_":
            j = i
            while j < len(codigo) and (codigo[j].isalnum() or codigo[j] == "_"):
                j += 1
            palabra = codigo[i:j]
            tokens.append(Token(palabra if palabra in RESERVADAS else "NOMBRE", palabra, linea, columna))
            i = j
        elif c == '"':
            j, partes = i + 1, []
            while True:
                if j >= len(codigo) or codigo[j] == "\n":
                    raise ErrorLexico("texto sin cerrar", linea, columna)
                if codigo[j] == '"':
                    break
                if codigo[j] == "\\":
                    if j + 1 < len(codigo) and codigo[j + 1] in ESCAPES:
                        partes.append(ESCAPES[codigo[j + 1]])
                        j += 2
                        continue
                    raise ErrorLexico("secuencia de escape no válida", linea, j - inicio_linea + 1)
                partes.append(codigo[j])
                j += 1
            tokens.append(Token("TEXTO", "".join(partes), linea, columna))
            i = j + 1
        else:
            for s in SIMBOLOS:
                if codigo.startswith(s, i):
                    tokens.append(Token(s, s, linea, columna))
                    i += len(s)
                    break
            else:
                raise ErrorLexico(f"carácter inesperado {c!r}", linea, columna)
    tokens.append(Token("FIN", None, linea, i - inicio_linea + 1))
    return tokens


codigo = """sea año = 2026;
si (año >= 1496) {
    año = año - 1;
}"""
tokens = tokenizar(codigo)
for t in tokens:
    print(f"{t.linea}:{t.columna:<3} {t.tipo:<8} {t.valor!r}")

lineas = codigo.splitlines()
for t in tokens[:-1]:
    assert lineas[t.linea - 1][t.columna - 1:].startswith(str(t.valor)), t

1:1   sea      'sea'
1:5   NOMBRE   'año'
1:9   =        '='
1:11  NUMERO   2026
1:15  ;        ';'
2:1   si       'si'
2:4   (        '('
2:5   NOMBRE   'año'
2:9   >=       '>='
2:12  NUMERO   1496
2:16  )        ')'
2:18  {        '{'
3:5   NOMBRE   'año'
3:9   =        '='
3:11  NOMBRE   'año'
3:15  -        '-'
3:17  NUMERO   1
3:18  ;        ';'
4:1   }        '}'
4:2   FIN      None


## 2. Comentarios, textos y palabras reservadas

Ampliar las pruebas con comentarios (desde `#` hasta el final de la línea), textos entre comillas con las secuencias de escape `\n`, `\t`, `\"` y `\\`, y palabras reservadas frente a nombres que solo empiezan como ellas (`sino` y `sinos`, `y` y `ya`). Mostrar los tipos de token de un pequeño programa.

In [2]:
programa = r'''# saludo desde Teror
sea mensaje = "Hola, \"Teror\"\n";   # texto con escapes
sea sinos = verdadero y no falso;
ya = nulo;'''
tokens = tokenizar(programa)
print(" ".join(t.tipo for t in tokens))
texto = next(t for t in tokens if t.tipo == "TEXTO")
print("Texto leído:", repr(texto.valor))

assert texto.valor == 'Hola, "Teror"\n'
assert [t.tipo for t in tokens if t.valor in ("sinos", "ya")] == ["NOMBRE", "NOMBRE"]
assert [t.tipo for t in tokens if t.tipo in RESERVADAS] == ["sea", "sea", "verdadero", "y", "no", "falso", "nulo"]
assert all(t.linea > 1 for t in tokens)

sea NOMBRE = TEXTO ; sea NOMBRE = verdadero y no falso ; NOMBRE = nulo ; FIN
Texto leído: 'Hola, "Teror"\n'


## 3. Errores léxicos con línea y columna

Escribir `marcar_error(codigo, error)`, que muestre el mensaje, la línea del código y un acento circunflejo bajo la columna del error. Probarlo con un carácter que no es del lenguaje, un texto sin cerrar, un número pegado a letras y un escape inválido, comprobando la posición de cada error.

In [3]:
def marcar_error(codigo, error):
    lineas = codigo.splitlines() + [""]
    print(f"  error {error}")
    print(f"    {lineas[error.linea - 1]}")
    print("    " + " " * (error.columna - 1) + "^")


casos = [
    ('sea precio = 3 @ 4;', (1, 16)),
    ('sea x = 1;\nescribe("Agaete);', (2, 9)),
    ('sea total = 12guaguas;', (1, 13)),
    ('sea t = "Gáldar\\q";', (1, 16)),
]
for fuente, esperado in casos:
    try:
        tokenizar(fuente)
    except ErrorLexico as e:
        marcar_error(fuente, e)
        assert (e.linea, e.columna) == esperado
    else:
        raise AssertionError("debía fallar")

  error 1:16: carácter inesperado '@'
    sea precio = 3 @ 4;
                   ^
  error 2:9: texto sin cerrar
    escribe("Agaete);
            ^
  error 1:13: número mal formado '12g'
    sea total = 12guaguas;
                ^
  error 1:16: secuencia de escape no válida
    sea t = "Gáldar\q";
                   ^


## 4. Segundo lexer con expresiones regulares

Escribir `tokenizar_re(codigo)` con un único patrón de grupos con nombre (`re.VERBOSE` y `lastgroup`) y comprobar que produce exactamente los mismos tokens que el lexer a mano en varios programas, y los errores en la misma posición en los casos del ejercicio anterior que no dependen del escape.

In [4]:
import re

PATRON = re.compile(r"""
    (?P<ESPACIO>[ \t\r]+) | (?P<SALTO>\n) | (?P<COMENTARIO>\#[^\n]*)
  | (?P<MAL_NUMERO>[0-9]+[^\W0-9]) | (?P<NUMERO>[0-9]+)
  | (?P<NOMBRE>[^\W0-9]\w*)
  | (?P<TEXTO>"(?:[^"\\\n]|\\[nt"\\])*")
  | (?P<SIMBOLO>==|!=|<=|>=|[-+*/%^<>=(){},;])
  | (?P<ERROR>.)
""", re.VERBOSE)


def tokenizar_re(codigo):
    tokens, linea, inicio = [], 1, 0
    for m in PATRON.finditer(codigo):
        tipo, texto, columna = m.lastgroup, m.group(), m.start() - inicio + 1
        match tipo:
            case "SALTO":
                linea, inicio = linea + 1, m.end()
            case "ESPACIO" | "COMENTARIO":
                pass
            case "MAL_NUMERO":
                raise ErrorLexico(f"número mal formado {texto!r}", linea, columna)
            case "NUMERO":
                tokens.append(Token("NUMERO", int(texto), linea, columna))
            case "NOMBRE":
                tokens.append(Token(texto if texto in RESERVADAS else "NOMBRE", texto, linea, columna))
            case "TEXTO":
                valor = re.sub(r"\\(.)", lambda e: ESCAPES[e[1]], texto[1:-1])
                tokens.append(Token("TEXTO", valor, linea, columna))
            case "SIMBOLO":
                tokens.append(Token(texto, texto, linea, columna))
            case "ERROR":
                mensaje = "texto sin cerrar" if texto == '"' else f"carácter inesperado {texto!r}"
                raise ErrorLexico(mensaje, linea, columna)
    tokens.append(Token("FIN", None, linea, len(codigo) - inicio + 1))
    return tokens


FACTORIAL = """# factorial recursivo
funcion factorial(n) {
    si (n <= 1) { devuelve 1; }
    devuelve n * factorial(n - 1);
}
escribe("5! =", factorial(5));
"""
FIZZBUZZ = """sea i = 1;
mientras (i <= 15) {
    si (i % 15 == 0) { escribe("FizzBuzz"); }
    sino si (i % 3 == 0) { escribe("Fizz"); }
    sino si (i % 5 == 0) { escribe("Buzz"); }
    sino { escribe(i); }
    i = i + 1;
}
"""
for fuente in [codigo, programa, FACTORIAL, FIZZBUZZ, "", "   # solo un comentario"]:
    assert tokenizar(fuente) == tokenizar_re(fuente)
print("Mismos tokens en 6 programas; FizzBuzz tiene", len(tokenizar(FIZZBUZZ)), "tokens")

for fuente, _ in casos[:3]:
    errores = []
    for lexer in (tokenizar, tokenizar_re):
        try:
            lexer(fuente)
        except ErrorLexico as e:
            errores.append((e.linea, e.columna))
    print(f"{fuente!r:32} → {errores}")
    assert len(errores) == 2 and errores[0] == errores[1]

Mismos tokens en 6 programas; FizzBuzz tiene 75 tokens
'sea precio = 3 @ 4;'            → [(1, 16), (1, 16)]
'sea x = 1;\nescribe("Agaete);'  → [(2, 9), (2, 9)]
'sea total = 12guaguas;'         → [(1, 13), (1, 13)]


## 5. Gramática en EBNF y expresiones aleatorias

Escribir la gramática de Guanche en EBNF, de menor a mayor precedencia, y un generador de expresiones aleatorias que siga sus reglas de expresión (con semilla fija) para usarlo en las pruebas de los analizadores. Las comparaciones no se encadenan y `^` es asociativo por la derecha, como `**` en Python.

In [5]:
import random

GRAMATICA = """
programa    = { sentencia } ;
sentencia   = "sea" NOMBRE "=" expresion ";"
            | NOMBRE "=" expresion ";"
            | si
            | "mientras" "(" expresion ")" bloque
            | "funcion" NOMBRE "(" [ NOMBRE { "," NOMBRE } ] ")" bloque
            | "devuelve" [ expresion ] ";"
            | expresion ";" ;
si          = "si" "(" expresion ")" bloque [ "sino" ( bloque | si ) ] ;
bloque      = "{" { sentencia } "}" ;
expresion   = conjuncion { "o" conjuncion } ;
conjuncion  = negacion { "y" negacion } ;
negacion    = "no" negacion | comparacion ;
comparacion = suma [ ( "==" | "!=" | "<" | "<=" | ">" | ">=" ) suma ] ;
suma        = producto { ( "+" | "-" ) producto } ;
producto    = unario { ( "*" | "/" | "%" ) unario } ;
unario      = "-" unario | potencia ;
potencia    = llamada [ "^" unario ] ;
llamada     = primario { "(" [ expresion { "," expresion } ] ")" } ;
primario    = NUMERO | TEXTO | "verdadero" | "falso" | "nulo" | NOMBRE | "(" expresion ")" ;
"""
print(GRAMATICA)


def expresion_aleatoria(rng, profundidad=3, variables=(), funciones=(), logica=False):
    def primario(p):
        opciones = ["numero"] * 3 + ["variable"] * bool(variables) * 2
        if p > 0:
            opciones += ["parentesis"] * 2 + ["llamada"] * bool(funciones)
        match rng.choice(opciones):
            case "numero":
                return str(rng.randint(0, 20))
            case "variable":
                return rng.choice(variables)
            case "parentesis":
                return f"({inicial(p - 1)})"
            case "llamada":
                args = ", ".join(inicial(p - 1) for _ in range(rng.randint(0, 2)))
                return f"{rng.choice(funciones)}({args})"

    def potencia(p):
        base = primario(p)
        if rng.random() < 0.15:
            exponente = str(rng.randint(0, 3)) + (f" ^ {rng.randint(0, 2)}" if rng.random() < 0.3 else "")
            return f"{base} ^ {exponente}"
        return base

    def unario(p):
        return ("-" if rng.random() < 0.2 else "") + potencia(p)

    def cadena(operando, operadores, p, prob):
        partes = [operando(p)]
        while rng.random() < prob and len(partes) < 7:
            partes += [rng.choice(operadores), operando(p)]
        return " ".join(partes)

    def suma(p):
        return cadena(lambda q: cadena(unario, ["*", "/", "%"], q, 0.4), ["+", "-"], p, 0.5)

    def comparacion(p):
        if rng.random() < 0.4:
            return f"{suma(p)} {rng.choice(['==', '!=', '<', '<=', '>', '>='])} {suma(p)}"
        return suma(p)

    def negacion(p):
        return ("no " if rng.random() < 0.2 else "") + comparacion(p)

    def disyuncion(p):
        return cadena(lambda q: cadena(negacion, ["y"], q, 0.3), ["o"], p, 0.3)

    inicial = disyuncion if logica else suma
    return inicial(profundidad)


rng = random.Random(7)
for _ in range(3):
    print(expresion_aleatoria(rng, profundidad=2, variables=("a", "b")))
for _ in range(3):
    print(expresion_aleatoria(rng, profundidad=1, variables=("x",), funciones=("f",), logica=True))
for _ in range(200):
    tokenizar(expresion_aleatoria(rng, variables=("a",), funciones=("f",), logica=True))


programa    = { sentencia } ;
sentencia   = "sea" NOMBRE "=" expresion ";"
            | NOMBRE "=" expresion ";"
            | si
            | "mientras" "(" expresion ")" bloque
            | "funcion" NOMBRE "(" [ NOMBRE { "," NOMBRE } ] ")" bloque
            | "devuelve" [ expresion ] ";"
            | expresion ";" ;
si          = "si" "(" expresion ")" bloque [ "sino" ( bloque | si ) ] ;
bloque      = "{" { sentencia } "}" ;
expresion   = conjuncion { "o" conjuncion } ;
conjuncion  = negacion { "y" negacion } ;
negacion    = "no" negacion | comparacion ;
comparacion = suma [ ( "==" | "!=" | "<" | "<=" | ">" | ">=" ) suma ] ;
suma        = producto { ( "+" | "-" ) producto } ;
producto    = unario { ( "*" | "/" | "%" ) unario } ;
unario      = "-" unario | potencia ;
potencia    = llamada [ "^" unario ] ;
llamada     = primario { "(" [ expresion { "," expresion } ] ")" } ;
primario    = NUMERO | TEXTO | "verdadero" | "falso" | "nulo" | NOMBRE | "(" expresion ")" ;

12 % -a * 7 

## 6. Árbol de sintaxis abstracta con `dataclasses`

Definir los nodos del AST como `dataclasses` inmutables (`frozen=True`, `slots=True`): literales, `Variable`, `Unaria`, `Binaria`, `Llamada` y las sentencias. Cada nodo guarda su posición en un campo `pos` que no participa en la igualdad, de modo que dos árboles iguales escritos en sitios distintos se comparan como iguales. Escribir `como_s(nodo)`, que muestre una expresión como expresión S (`(+ 1 (* 2 x))`).

In [6]:
from dataclasses import field, FrozenInstanceError

nodo = dataclass(frozen=True, slots=True)


def posicion():
    return field(default=(0, 0), compare=False, repr=False)


class Nodo:
    __slots__ = ()


@nodo
class Numero(Nodo):
    valor: int
    pos: tuple = posicion()

@nodo
class Texto(Nodo):
    valor: str
    pos: tuple = posicion()

@nodo
class Logico(Nodo):
    valor: bool
    pos: tuple = posicion()

@nodo
class Nulo(Nodo):
    pos: tuple = posicion()

@nodo
class Variable(Nodo):
    nombre: str
    pos: tuple = posicion()

@nodo
class Unaria(Nodo):
    op: str
    operando: Nodo
    pos: tuple = posicion()

@nodo
class Binaria(Nodo):
    op: str
    izq: Nodo
    der: Nodo
    pos: tuple = posicion()

@nodo
class Llamada(Nodo):
    funcion: Nodo
    argumentos: tuple
    pos: tuple = posicion()

@nodo
class Declaracion(Nodo):
    nombre: str
    valor: Nodo
    pos: tuple = posicion()

@nodo
class Asignacion(Nodo):
    nombre: str
    valor: Nodo
    pos: tuple = posicion()

@nodo
class Bloque(Nodo):
    sentencias: tuple
    pos: tuple = posicion()

@nodo
class Si(Nodo):
    condicion: Nodo
    entonces: Bloque
    sino: Bloque | None
    pos: tuple = posicion()

@nodo
class Mientras(Nodo):
    condicion: Nodo
    cuerpo: Bloque
    pos: tuple = posicion()

@nodo
class Funcion(Nodo):
    nombre: str
    parametros: tuple
    cuerpo: Bloque
    pos: tuple = posicion()

@nodo
class Devuelve(Nodo):
    valor: Nodo | None
    pos: tuple = posicion()

@nodo
class ExprSentencia(Nodo):
    expresion: Nodo
    pos: tuple = posicion()

@nodo
class Programa(Nodo):
    sentencias: tuple
    pos: tuple = posicion()


def como_s(n):
    match n:
        case Numero(valor=v):
            return str(v)
        case Texto(valor=v):
            return repr(v)
        case Logico(valor=v):
            return "verdadero" if v else "falso"
        case Nulo():
            return "nulo"
        case Variable(nombre=nombre):
            return nombre
        case Unaria(op=op, operando=x):
            return f"({op} {como_s(x)})"
        case Binaria(op=op, izq=a, der=b):
            return f"({op} {como_s(a)} {como_s(b)})"
        case Llamada(funcion=f, argumentos=args):
            return "(" + " ".join(["llamar", como_s(f), *map(como_s, args)]) + ")"


arbol = Binaria("+", Numero(1, pos=(1, 1)), Binaria("*", Numero(2), Variable("x")), pos=(1, 3))
print(como_s(arbol))
print(arbol)
assert arbol == Binaria("+", Numero(1), Binaria("*", Numero(2), Variable("x")))
try:
    arbol.op = "-"
except FrozenInstanceError as e:
    print("Inmutable:", type(e).__name__, e)

(+ 1 (* 2 x))
Binaria(op='+', izq=Numero(valor=1), der=Binaria(op='*', izq=Numero(valor=2), der=Variable(nombre='x')))
Inmutable: FrozenInstanceError cannot assign to field 'op'


## 7. Analizador descendente recursivo de expresiones

Escribir `AnalizadorExpresiones`, con un método por cada regla de la gramática (`disyuncion`, `conjuncion`, …, `primario`) y los auxiliares `acepta` y `espera`. Comprobarlo con 400 expresiones aleatorias: traducidas a Python (`^` → `**`, `/` → `//`), el árbol de `ast.parse` debe tener la misma forma que el de Guanche, porque ambas gramáticas tienen la misma precedencia y asociatividad aritmética.

In [7]:
import ast

COMPARACIONES = ("==", "!=", "<", "<=", ">", ">=")


class ErrorSintactico(Exception):
    def __init__(self, mensaje, token):
        super().__init__(f"{token.linea}:{token.columna}: {mensaje}")
        self.linea, self.columna = token.linea, token.columna


def describir(token):
    return "el final del programa" if token.tipo == "FIN" else repr(str(token.valor))


def p(token):
    return (token.linea, token.columna)


class AnalizadorExpresiones:
    def __init__(self, codigo):
        self.tokens = tokenizar(codigo)
        self.i = 0

    @property
    def actual(self):
        return self.tokens[self.i]

    def siguiente(self):
        return self.tokens[min(self.i + 1, len(self.tokens) - 1)]

    def avanzar(self):
        t = self.tokens[self.i]
        if t.tipo != "FIN":
            self.i += 1
        return t

    def acepta(self, *tipos):
        return self.avanzar() if self.actual.tipo in tipos else None

    def espera(self, tipo, que=None):
        if self.actual.tipo != tipo:
            raise ErrorSintactico(f"se esperaba {que or repr(tipo)} y se encontró {describir(self.actual)}", self.actual)
        return self.avanzar()

    def solo_expresion(self):
        e = self.expresion()
        self.espera("FIN", "el final")
        return e

    def expresion(self):
        return self.disyuncion()

    def disyuncion(self):
        izq = self.conjuncion()
        while t := self.acepta("o"):
            izq = Binaria("o", izq, self.conjuncion(), pos=p(t))
        return izq

    def conjuncion(self):
        izq = self.negacion()
        while t := self.acepta("y"):
            izq = Binaria("y", izq, self.negacion(), pos=p(t))
        return izq

    def negacion(self):
        if t := self.acepta("no"):
            return Unaria("no", self.negacion(), pos=p(t))
        return self.comparacion()

    def comparacion(self):
        izq = self.suma()
        if t := self.acepta(*COMPARACIONES):
            izq = Binaria(t.tipo, izq, self.suma(), pos=p(t))
        return izq

    def suma(self):
        izq = self.producto()
        while t := self.acepta("+", "-"):
            izq = Binaria(t.tipo, izq, self.producto(), pos=p(t))
        return izq

    def producto(self):
        izq = self.unario()
        while t := self.acepta("*", "/", "%"):
            izq = Binaria(t.tipo, izq, self.unario(), pos=p(t))
        return izq

    def unario(self):
        if t := self.acepta("-"):
            return Unaria("-", self.unario(), pos=p(t))
        return self.potencia()

    def potencia(self):
        base = self.llamada()
        if t := self.acepta("^"):
            return Binaria("^", base, self.unario(), pos=p(t))
        return base

    def argumentos(self):
        args = []
        if self.actual.tipo != ")":
            args.append(self.expresion())
            while self.acepta(","):
                args.append(self.expresion())
        self.espera(")")
        return tuple(args)

    def llamada(self):
        e = self.primario()
        while t := self.acepta("("):
            e = Llamada(e, self.argumentos(), pos=p(t))
        return e

    def primario(self):
        t = self.actual
        match t.tipo:
            case "NUMERO":
                return Numero(self.avanzar().valor, pos=p(t))
            case "TEXTO":
                return Texto(self.avanzar().valor, pos=p(t))
            case "verdadero" | "falso":
                return Logico(self.avanzar().tipo == "verdadero", pos=p(t))
            case "nulo":
                self.avanzar()
                return Nulo(pos=p(t))
            case "NOMBRE":
                return Variable(self.avanzar().valor, pos=p(t))
            case "(":
                self.avanzar()
                e = self.expresion()
                self.espera(")")
                return e
        raise ErrorSintactico(f"se esperaba una expresión y se encontró {describir(t)}", t)


def analizar_expresion(codigo):
    return AnalizadorExpresiones(codigo).solo_expresion()


OPS_PYTHON = {ast.Add: "+", ast.Sub: "-", ast.Mult: "*", ast.FloorDiv: "/", ast.Mod: "%", ast.Pow: "^"}


def s_python(n):
    match n:
        case ast.BinOp(left=a, op=op, right=b):
            return f"({OPS_PYTHON[type(op)]} {s_python(a)} {s_python(b)})"
        case ast.UnaryOp(op=ast.USub(), operand=x):
            return f"(- {s_python(x)})"
        case ast.Constant(value=v):
            return str(v)
        case ast.Name(id=nombre):
            return nombre


def a_python(expresion):
    return expresion.replace("^", "**").replace("/", "//")


ejemplo = "1 + 2 * -x ^ 2 - (z - 3) / 4"
print(ejemplo, "→", como_s(analizar_expresion(ejemplo)))
print(a_python(ejemplo), "→", s_python(ast.parse(a_python(ejemplo), mode="eval").body))

rng = random.Random(2026)
for _ in range(400):
    e = expresion_aleatoria(rng, profundidad=4, variables=("x", "z"))
    assert como_s(analizar_expresion(e)) == s_python(ast.parse(a_python(e), mode="eval").body), e
print("400 expresiones con la misma forma que en Python")

1 + 2 * -x ^ 2 - (z - 3) / 4 → (- (+ 1 (* 2 (- (^ x 2)))) (/ (- z 3) 4))
1 + 2 * -x ** 2 - (z - 3) // 4 → (- (+ 1 (* 2 (- (^ x 2)))) (/ (- z 3) 4))
400 expresiones con la misma forma que en Python


## 8. Precedencia y asociatividad

Mostrar en una tabla cómo agrupa el analizador casos que suelen confundirse: resta y división por la izquierda, potencia por la derecha, el menos unario frente a la potencia (`-2 ^ 2`), la potencia con exponente negativo, `no` frente a `y` y `o`, y llamadas encadenadas. Comprobar cada agrupación y que las comparaciones encadenadas (`a < b < c`) se rechazan.

In [8]:
casos = {
    "10 - 4 - 3": "(- (- 10 4) 3)",
    "64 / 4 / 2": "(/ (/ 64 4) 2)",
    "2 ^ 3 ^ 2": "(^ 2 (^ 3 2))",
    "-2 ^ 2": "(- (^ 2 2))",
    "2 ^ -1": "(^ 2 (- 1))",
    "-a * b": "(* (- a) b)",
    "a + b * c ^ d": "(+ a (* b (^ c d)))",
    "no a y b o c": "(o (y (no a) b) c)",
    "no a == b": "(no (== a b))",
    "a o b y c": "(o a (y b c))",
    "f(1)(2, x)": "(llamar (llamar f 1) 2 x)",
    "-f(x) ^ 2": "(- (^ (llamar f x) 2))",
}
for fuente, esperado in casos.items():
    obtenido = como_s(analizar_expresion(fuente))
    print(f"{fuente:16} {obtenido}")
    assert obtenido == esperado

try:
    analizar_expresion("a < b < c")
except ErrorSintactico as e:
    print("a < b < c →", e)

10 - 4 - 3       (- (- 10 4) 3)
64 / 4 / 2       (/ (/ 64 4) 2)
2 ^ 3 ^ 2        (^ 2 (^ 3 2))
-2 ^ 2           (- (^ 2 2))
2 ^ -1           (^ 2 (- 1))
-a * b           (* (- a) b)
a + b * c ^ d    (+ a (* b (^ c d)))
no a y b o c     (o (y (no a) b) c)
no a == b        (no (== a b))
a o b y c        (o a (y b c))
f(1)(2, x)       (llamar (llamar f 1) 2 x)
-f(x) ^ 2        (- (^ (llamar f x) 2))
a < b < c → 1:7: se esperaba el final y se encontró '<'


## 9. Sentencias, bloques y programas

Ampliar el analizador con las sentencias en `AnalizadorGuanche`: declaración, asignación (que necesita mirar un token por delante para distinguir `x = 1;` de `x;`), `si`/`sino` (con `sino si` encadenado), `mientras`, `funcion` y `devuelve`. Escribir `mostrar_arbol(nodo)`, que dibuje el árbol con sangría, y aplicarlo al factorial.

In [9]:
from dataclasses import fields


class AnalizadorGuanche(AnalizadorExpresiones):
    def programa(self):
        sentencias = []
        while self.actual.tipo != "FIN":
            sentencias.append(self.sentencia())
        return Programa(tuple(sentencias))

    def bloque(self):
        t = self.espera("{")
        sentencias = []
        while self.actual.tipo not in ("}", "FIN"):
            sentencias.append(self.sentencia())
        self.espera("}")
        return Bloque(tuple(sentencias), pos=p(t))

    def sentencia(self):
        t = self.actual
        match t.tipo:
            case "sea":
                self.avanzar()
                nombre = self.espera("NOMBRE", "un nombre").valor
                self.espera("=")
                valor = self.expresion()
                self.espera(";")
                return Declaracion(nombre, valor, pos=p(t))
            case "NOMBRE" if self.siguiente().tipo == "=":
                self.avanzar()
                self.avanzar()
                valor = self.expresion()
                self.espera(";")
                return Asignacion(t.valor, valor, pos=p(t))
            case "si":
                return self.si()
            case "mientras":
                self.avanzar()
                self.espera("(")
                condicion = self.expresion()
                self.espera(")")
                return Mientras(condicion, self.bloque(), pos=p(t))
            case "funcion":
                self.avanzar()
                nombre = self.espera("NOMBRE", "el nombre de la función").valor
                self.espera("(")
                parametros = []
                if self.actual.tipo != ")":
                    parametros.append(self.espera("NOMBRE", "un parámetro").valor)
                    while self.acepta(","):
                        parametros.append(self.espera("NOMBRE", "un parámetro").valor)
                self.espera(")")
                return Funcion(nombre, tuple(parametros), self.bloque(), pos=p(t))
            case "devuelve":
                self.avanzar()
                valor = None if self.actual.tipo == ";" else self.expresion()
                self.espera(";")
                return Devuelve(valor, pos=p(t))
        e = self.expresion()
        self.espera(";")
        return ExprSentencia(e, pos=p(t))

    def si(self):
        t = self.espera("si")
        self.espera("(")
        condicion = self.expresion()
        self.espera(")")
        entonces = self.bloque()
        sino = None
        if s := self.acepta("sino"):
            sino = Bloque((self.si(),), pos=p(s)) if self.actual.tipo == "si" else self.bloque()
        return Si(condicion, entonces, sino, pos=p(t))


def analizar(codigo):
    return AnalizadorGuanche(codigo).programa()


def mostrar_arbol(n, sangria="", etiqueta=""):
    if not isinstance(n, Nodo):
        print(f"{sangria}{etiqueta}{n!r}")
        return
    def es_subarbol(v):
        return isinstance(v, Nodo) or (isinstance(v, tuple) and any(isinstance(x, Nodo) for x in v))

    simples = {f.name: getattr(n, f.name) for f in fields(n) if f.name != "pos" and not es_subarbol(getattr(n, f.name))}
    texto = ", ".join(f"{k}={v!r}" for k, v in simples.items())
    print(f"{sangria}{etiqueta}{type(n).__name__}({texto})  @{n.pos[0]}:{n.pos[1]}")
    for f in fields(n):
        v = getattr(n, f.name)
        if isinstance(v, Nodo):
            mostrar_arbol(v, sangria + "  ", f.name + ": ")
        elif f.name != "pos" and es_subarbol(v):
            for x in v:
                mostrar_arbol(x, sangria + "  ", "· ")


arbol_factorial = analizar(FACTORIAL)
mostrar_arbol(arbol_factorial)

fizz = analizar(FIZZBUZZ)
cuerpo = fizz.sentencias[1].cuerpo.sentencias
assert isinstance(cuerpo[0], Si) and isinstance(cuerpo[0].sino.sentencias[0], Si)
assert isinstance(cuerpo[1], Asignacion)
assert analizar("x;").sentencias[0] == ExprSentencia(Variable("x"))

Programa()  @0:0
  · Funcion(nombre='factorial', parametros=('n',))  @2:1
    cuerpo: Bloque()  @2:22
      · Si(sino=None)  @3:5
        condicion: Binaria(op='<=')  @3:11
          izq: Variable(nombre='n')  @3:9
          der: Numero(valor=1)  @3:14
        entonces: Bloque()  @3:17
          · Devuelve()  @3:19
            valor: Numero(valor=1)  @3:28
      · Devuelve()  @4:5
        valor: Binaria(op='*')  @4:16
          izq: Variable(nombre='n')  @4:14
          der: Llamada()  @4:27
            funcion: Variable(nombre='factorial')  @4:18
            · Binaria(op='-')  @4:30
              izq: Variable(nombre='n')  @4:28
              der: Numero(valor=1)  @4:32
  · ExprSentencia()  @6:1
    expresion: Llamada()  @6:8
      funcion: Variable(nombre='escribe')  @6:1
      · Texto(valor='5! =')  @6:9
      · Llamada()  @6:26
        funcion: Variable(nombre='factorial')  @6:17
        · Numero(valor=5)  @6:27


## 10. Errores sintácticos con línea y columna

Hacer que cada error sintáctico diga qué se esperaba, qué se encontró y dónde, y mostrarlo con `marcar_error`. Probar un punto y coma olvidado, un paréntesis sin cerrar, un `sino` sin `si`, un bloque sin cerrar al final del programa, una coma de más en una llamada y una comparación encadenada.

In [10]:
casos = [
    ("sea x = 1\nsea y = 2;", (2, 1)),
    ("si (x > 1 { x = 0; }", (1, 11)),
    ("sea z = 3;\nsino { z = 4; }", (2, 1)),
    ("mientras (verdadero) {\n    x = x + 1;\n", (3, 1)),
    ("escribe(1, , 2);", (1, 12)),
    ("si (0 < x < 9) { }", (1, 11)),
]
for fuente, esperado in casos:
    try:
        analizar(fuente)
    except ErrorSintactico as e:
        marcar_error(fuente, e)
        assert (e.linea, e.columna) == esperado, (e.linea, e.columna)
    else:
        raise AssertionError(fuente)

  error 2:1: se esperaba ';' y se encontró 'sea'
    sea y = 2;
    ^
  error 1:11: se esperaba ')' y se encontró '{'
    si (x > 1 { x = 0; }
              ^
  error 2:1: se esperaba una expresión y se encontró 'sino'
    sino { z = 4; }
    ^
  error 3:1: se esperaba '}' y se encontró el final del programa
    
    ^
  error 1:12: se esperaba una expresión y se encontró ','
    escribe(1, , 2);
               ^
  error 1:11: se esperaba ')' y se encontró '<'
    si (0 < x < 9) { }
              ^


## 11. Analizador de Pratt

Escribir `AnalizadorPratt`, que analiza las expresiones con una tabla de *poderes de enlace* en lugar de un método por nivel: cada operador infijo tiene un poder izquierdo y la función `expresion(rbp)` sigue consumiendo operadores mientras su poder supere `rbp`. La asociatividad por la derecha de `^` se consigue bajando en uno el poder del operando derecho. Comprobar que produce el mismo AST que el descendente recursivo en 1000 expresiones aleatorias con lógica, comparaciones y llamadas.

In [11]:
PODER = {"o": 1, "y": 2, **dict.fromkeys(COMPARACIONES, 4), "+": 5, "-": 5, "*": 6, "/": 6, "%": 6, "^": 8, "(": 10}
PODER_NO, PODER_MENOS = 3, 7


class AnalizadorPratt(AnalizadorExpresiones):
    def expresion(self, rbp=0):
        izq = self.prefijo(self.avanzar())
        while PODER.get(self.actual.tipo, 0) > rbp:
            izq = self.infijo(self.avanzar(), izq)
        return izq

    def prefijo(self, t):
        match t.tipo:
            case "-":
                return Unaria("-", self.expresion(PODER_MENOS), pos=p(t))
            case "no":
                return Unaria("no", self.expresion(PODER_NO), pos=p(t))
            case "(":
                e = self.expresion()
                self.espera(")")
                return e
        self.i -= t.tipo != "FIN"
        return self.primario()

    def infijo(self, t, izq):
        match t.tipo:
            case "(":
                return Llamada(izq, self.argumentos(), pos=p(t))
            case "^":
                return Binaria("^", izq, self.expresion(PODER["^"] - 1), pos=p(t))
        der = self.expresion(PODER[t.tipo])
        if t.tipo in COMPARACIONES and self.actual.tipo in COMPARACIONES:
            raise ErrorSintactico("las comparaciones no se encadenan", self.actual)
        return Binaria(t.tipo, izq, der, pos=p(t))


def analizar_pratt(codigo):
    return AnalizadorPratt(codigo).solo_expresion()


for fuente in ["2 ^ 3 ^ 2", "no a == b y c", "-f(x)(z) ^ 2 * 3"]:
    print(f"{fuente:18} {como_s(analizar_pratt(fuente))}")

rng = random.Random(11)
for _ in range(1000):
    e = expresion_aleatoria(rng, profundidad=3, variables=("a", "b"), funciones=("f", "g"), logica=True)
    assert analizar_pratt(e) == analizar_expresion(e), e
for malo in ["a < b < c", "1 +", "(2", "f(1,)"]:
    for analizador in (analizar_pratt, analizar_expresion):
        try:
            analizador(malo)
        except ErrorSintactico:
            pass
        else:
            raise AssertionError(malo)
print("Pratt y descendente recursivo coinciden en 1000 expresiones y rechazan las mismas 4 incorrectas")

2 ^ 3 ^ 2          (^ 2 (^ 3 2))
no a == b y c      (y (no (== a b)) c)
-f(x)(z) ^ 2 * 3   (* (- (^ (llamar (llamar f x) z) 2)) 3)


Pratt y descendente recursivo coinciden en 1000 expresiones y rechazan las mismas 4 incorrectas


## 12. Del árbol al código con los paréntesis justos

Escribir `a_codigo(expresion)` e `imprimir(sentencia)`, que conviertan el AST de vuelta en código Guanche con sangría y poniendo paréntesis solo cuando la precedencia o la asociatividad los exigen. Comprobar la ida y vuelta: analizar el código impreso debe dar el mismo árbol, en 1000 expresiones aleatorias y en los programas de ejemplo. Comparar el número de paréntesis con el de la expresión original.

In [12]:
PREC = {"o": 1, "y": 2, **dict.fromkeys(COMPARACIONES, 4), "+": 5, "-": 5, "*": 6, "/": 6, "%": 6, "^": 8}


def prec(n):
    match n:
        case Binaria(op=op):
            return PREC[op]
        case Unaria(op="no"):
            return PODER_NO
        case Unaria():
            return PODER_MENOS
        case Llamada():
            return 10
    return 11


def envolver(n, hace_falta):
    return f"({a_codigo(n)})" if hace_falta else a_codigo(n)


def a_codigo(n):
    match n:
        case Numero(valor=v):
            return str(v)
        case Texto(valor=v):
            return '"' + v.replace("\\", "\\\\").replace('"', '\\"').replace("\n", "\\n").replace("\t", "\\t") + '"'
        case Logico(valor=v):
            return "verdadero" if v else "falso"
        case Nulo():
            return "nulo"
        case Variable(nombre=nombre):
            return nombre
        case Unaria(op="no", operando=x):
            return "no " + envolver(x, prec(x) < PODER_NO)
        case Unaria(op="-", operando=x):
            return "-" + envolver(x, prec(x) < PODER_MENOS)
        case Binaria(op="^", izq=a, der=b):
            return f"{envolver(a, prec(a) < 10)} ^ {envolver(b, prec(b) < PODER_MENOS)}"
        case Binaria(op=op, izq=a, der=b):
            no_asociativo = op in COMPARACIONES
            return f"{envolver(a, prec(a) < PREC[op] or (no_asociativo and prec(a) == PREC[op]))} {op} {envolver(b, prec(b) <= PREC[op])}"
        case Llamada(funcion=f, argumentos=args):
            return f"{envolver(f, prec(f) < 10)}({', '.join(map(a_codigo, args))})"


def bloque_codigo(b, nivel):
    if not b.sentencias:
        return "{ }"
    dentro = "\n".join(imprimir(s, nivel + 1) for s in b.sentencias)
    return "{\n" + dentro + "\n" + "    " * nivel + "}"


def imprimir(n, nivel=0):
    sangria = "    " * nivel
    match n:
        case Programa(sentencias=s):
            return "\n".join(imprimir(x) for x in s)
        case Declaracion(nombre=nombre, valor=v):
            return f"{sangria}sea {nombre} = {a_codigo(v)};"
        case Asignacion(nombre=nombre, valor=v):
            return f"{sangria}{nombre} = {a_codigo(v)};"
        case ExprSentencia(expresion=e):
            return f"{sangria}{a_codigo(e)};"
        case Devuelve(valor=None):
            return f"{sangria}devuelve;"
        case Devuelve(valor=v):
            return f"{sangria}devuelve {a_codigo(v)};"
        case Si(condicion=c, entonces=a, sino=b):
            return f"{sangria}si ({a_codigo(c)}) {bloque_codigo(a, nivel)}" + (f" sino {bloque_codigo(b, nivel)}" if b else "")
        case Mientras(condicion=c, cuerpo=cuerpo):
            return f"{sangria}mientras ({a_codigo(c)}) {bloque_codigo(cuerpo, nivel)}"
        case Funcion(nombre=nombre, parametros=ps, cuerpo=cuerpo):
            return f"{sangria}funcion {nombre}({', '.join(ps)}) {bloque_codigo(cuerpo, nivel)}"


feo = "((1 + (2 * x)) - ((-(z)) ^ (2)))  o  (no (a))"
print(feo, "→", a_codigo(analizar_expresion(feo)))
print()
print(imprimir(analizar(FACTORIAL)))

rng = random.Random(12)
parentesis_antes = parentesis_despues = 0
for _ in range(1000):
    e = expresion_aleatoria(rng, profundidad=4, variables=("a", "b"), funciones=("f",), logica=True)
    impresa = a_codigo(analizar_expresion(e))
    assert analizar_expresion(impresa) == analizar_expresion(e), (e, impresa)
    parentesis_antes += e.count("(")
    parentesis_despues += impresa.count("(")
for fuente in (FACTORIAL, FIZZBUZZ, programa, codigo):
    assert analizar(imprimir(analizar(fuente))) == analizar(fuente)
print(f"\nParéntesis en 1000 expresiones: {parentesis_antes} → {parentesis_despues}")
assert parentesis_despues <= parentesis_antes

((1 + (2 * x)) - ((-(z)) ^ (2)))  o  (no (a)) → 1 + 2 * x - (-z) ^ 2 o no a

funcion factorial(n) {
    si (n <= 1) {
        devuelve 1;
    }
    devuelve n * factorial(n - 1);
}
escribe("5! =", factorial(5));



Paréntesis en 1000 expresiones: 145752 → 134879


## 13. Recorridos del árbol

Escribir un recorrido genérico `hijos(nodo)` que use `dataclasses.fields` para obtener los subárboles de cualquier nodo, y con él contar los nodos de cada tipo y medir la profundidad del AST. Comprobar los totales con un recorrido iterativo con pila. Después, con `match`, calcular las *variables libres* de cada función (las que usa sin ser parámetros ni declaradas dentro), que son las que tendrá que capturar del entorno.

In [13]:
from collections import Counter


def hijos(n):
    for f in fields(n):
        v = getattr(n, f.name)
        if isinstance(v, Nodo):
            yield v
        elif isinstance(v, tuple) and f.name != "pos":
            yield from (x for x in v if isinstance(x, Nodo))


def recorrer(n):
    yield n
    for h in hijos(n):
        yield from recorrer(h)


def profundidad(n):
    return 1 + max((profundidad(h) for h in hijos(n)), default=0)


def recorrer_con_pila(raiz):
    pila = [raiz]
    while pila:
        n = pila.pop()
        yield n
        pila.extend(reversed(list(hijos(n))))


CONTADOR = """
funcion crear_contador(paso) {
    sea cuenta = 0;
    funcion siguiente() {
        cuenta = cuenta + paso;
        devuelve cuenta;
    }
    devuelve siguiente;
}
sea c = crear_contador(5);
c();
escribe(c());
"""
for nombre, fuente in [("factorial", FACTORIAL), ("fizzbuzz", FIZZBUZZ), ("contador", CONTADOR)]:
    arbol = analizar(fuente)
    tipos = Counter(type(n).__name__ for n in recorrer(arbol))
    print(f"{nombre:10} {sum(tipos.values()):3} nodos, profundidad {profundidad(arbol)}: {dict(tipos.most_common(4))}")
    assert list(recorrer(arbol)) == list(recorrer_con_pila(arbol))


def variables_libres(funcion):
    ligadas = set(funcion.parametros)
    usadas = set()

    def visitar(n):
        match n:
            case Declaracion(nombre=nombre) | Funcion(nombre=nombre):
                ligadas.add(nombre)
            case Variable(nombre=nombre) | Asignacion(nombre=nombre):
                usadas.add(nombre)
        if not isinstance(n, Funcion) or n is funcion:
            for h in hijos(n):
                visitar(h)

    visitar(funcion)
    return usadas - ligadas


for f in (n for n in recorrer(analizar(CONTADOR)) if isinstance(n, Funcion)):
    print(f"Variables libres de {f.nombre}: {sorted(variables_libres(f))}")
funciones = {f.nombre: f for f in recorrer(analizar(CONTADOR)) if isinstance(f, Funcion)}
assert variables_libres(funciones["siguiente"]) == {"cuenta", "paso"}
assert variables_libres(funciones["crear_contador"]) == set()

factorial   25 nodos, profundidad 8: {'Variable': 6, 'Numero': 4, 'Binaria': 3, 'Llamada': 3}
fizzbuzz    52 nodos, profundidad 12: {'Variable': 10, 'Numero': 9, 'Binaria': 8, 'Bloque': 7}
contador    27 nodos, profundidad 8: {'Variable': 8, 'Llamada': 4, 'Funcion': 2, 'Bloque': 2}
Variables libres de crear_contador: []
Variables libres de siguiente: ['cuenta', 'paso']
